# R-tag pipeline — single run walkthrough

This notebook runs the **Python R-tag (RTEG) workflow** for the KB331 sample filter.

**Manual reference:** Jing Yang's SKILL flow in [`../rdsBawTEGAutoFromTemp.il`](../rdsBawTEGAutoFromTemp.il)  
**Documentation:** [`../README.md`](../README.md) · [`../CLAUDE.md`](../CLAUDE.md)

Run all cells top-to-bottom from the `python_code/` directory.

| Step | Section | Source file(s) | Primary functions |
|------|---------|----------------|-------------------|
| 1 | Setup / inputs | `layermap.py` | `load_layermap` |
| 2.1 | Layermap | `layermap.py` | `load_layermap` |
| 2.2 | Inspect refs | `inspect_refs.py` | `inspect_gds` |
| 2.3 | Identify | `separate.py` | `identify` |
| 3 | PPD + orientation placement | `prep_resonator_ppd.py`, `rteg_orientation.py`, `rteg_collect.py` | `prep_resonator_ppd`, `analyze_orientation`, `preserved_collars_at_shift` |
| 4 | Die frame | `prep_rteg_frame.py`, `export_gds.py` | `prep_rteg_in_frame`, `export_gds` |
| 5.1 | Collect roles | `rteg_collect.py` | `collect_geometry_roles` |
| 5.2 | Classify / orientation | `rteg_orientation.py`, `rteg_classify.py` | `collect_orientation_inputs`, `classify_nodes` |
| 5.3 | MTE extensions | `rteg_mte_extensions.py` | `build_mte_extensions` |
| 5.4 | Route MTE (future) | TBD | TBD |
| 5 export | MTE GDS | `rteg_mte_extensions.py`, `export_gds.py` | `export_mte_extensions_gds` |


In [ ]:
from __future__ import annotations
import io
import sys
from contextlib import redirect_stdout
from pathlib import Path
import gdstk
import pandas as pd


def resolve_python_code_root() -> Path:
    """Find python_code/ by looking for input_files/ + src/."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "input_files").is_dir() and (candidate / "src").is_dir():
            return candidate
    return here


ROOT = resolve_python_code_root()
SRC = ROOT / "src"
INPUT_FILES = ROOT / "input_files"
DRAFT = ROOT / "draft_output"
ORIGNAL_RTEG = DRAFT / "original_rteg"

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

DRAFT.mkdir(parents=True, exist_ok=True)

print(f"Working directory: {ROOT}")
print(f"Input files:       {INPUT_FILES}")
print(f"Draft output:      {DRAFT}")
print(f"Source code:       {SRC}")

## Define Inputs Here For Running This Notebook

In [ ]:
FILTER = INPUT_FILES / "KB331_N_01.gds" # input filter GDS file
FRAME = INPUT_FILES / "KB331_N_Frame.gds" # die frame
PPD = INPUT_FILES / "GSG_frame.gds" # GSG ppd frame
LAYERMAP = INPUT_FILES / "layermap" # Skyworks layer map

## 1. Process Inputs

In [ ]:
# Ensure all referenced input files exist; abort on missing inputs

input_files = {
    "Filter layout": FILTER,
    "Frame template": FRAME,
    "Probe device": PPD,
    "Layer table": LAYERMAP,
}

input_roles = {
    "Filter layout": "Clean hierarchical filter GDS — resonators + connect metal",
    "Frame template": None,
    "Probe device": "ppd_1port — GSG pad reference",
    "Layer table": "Skyworks name → GDS (layer, datatype)",
}

rows = []
frame_size_str = "unknown size"

for label, path in input_files.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing required input: {label} ({path})")
    size = f"{path.stat().st_size:,} B"
    rows.append({"file": label, "path": path.name, "exists": True, "size": size, "role": input_roles[label]})

frame_lib = gdstk.read_gds(FRAME)
frame_cell = frame_lib.top_level()[0]
frame_bb = frame_cell.bounding_box()
if frame_bb is not None:
    (fx0, fy0), (fx1, fy1) = frame_bb
    frame_size_str = f"{fx1 - fx0:.1f}×{fy1 - fy0:.1f} µm"

for row in rows:
    if row["file"] == "Frame template":
        row["role"] = f"{frame_size_str} GSG probe frame (six BAW_MB2 pads)"

display(pd.DataFrame(rows))


## 2. Selection

2.1 start with layermap definitions

2.2 inspect layer references

2.3 start identifying resonators from filter GDS file

### 2.1 `layermap.py` — layer name lookups

Loads the Skyworks layermap file from `LAYERMAP` (defined above). Maps names like `BAW_MBE` to GDS `(layer, datatype)` pairs.

In [ ]:
from src.layermap import load_layermap

layermap = load_layermap(LAYERMAP)
print(f"Loaded {len(layermap)} layers from {LAYERMAP.name}")

for name in ("BAW_MBE", "BAW_MTE", "BAW_MB2", "BAW_EDGE"):
    layer, dt = layermap.pair(name)
    print(f"  {name:12s} -> GDS ({layer}, {dt})")

### 2.2 `inspect_refs.py` — hierarchy sanity check

Lists placed references in the filter GDS: resonators, vias, connect cells. Useful when instance names did not survive export.

In [ ]:
from src.inspect_refs import inspect_gds

buf = io.StringIO()
with redirect_stdout(buf):
    inspect_gds(FILTER)

# 
print(buf.getvalue()[:2000])
if len(buf.getvalue()) > 2000:
    print("... (truncated)")

### 2.3 `separate.py` — resonator and vtb identification

Finds placed resonators (masters: `series`, `shunt`, `rcap`, `mimcap`) and `vtb` vias under the filter parent cell. Returns dataframe-ready rows via `identify(FILTER)`.

In [ ]:
from src.separate import identify

identification = identify(FILTER)

parent = identification.parent
filter_lib = identification.library
res_list = identification.resonators
vias = identification.vias

res_df = pd.DataFrame(identification.resonator_rows()) #DF of resonators 
via_df = pd.DataFrame(identification.via_rows()) #DF of VIAs

print(f"Parent cell: {parent}")
print(f"Resonators: {len(res_list)}  |  Vias: {len(vias)}")

res_df

In [ ]:
via_df

## 3. Separation
For each resonator found place it together with the GSG ppd frame in the center

### 3 — PPD + orientation placement

**Files:** `src/prep_resonator_ppd.py`, `src/rteg_orientation.py`, `src/rteg_collect.py`

**Entry points:** `prep_resonator_ppd` (with `identification` + `layermap`) → `preserved_collars_at_shift` → `analyze_orientation` → extra `orientation_shift` on placement.

For each row in `res_df`, combine the resonator with the GSG PPD frame (`GSG_frame.gds`):
center on the template, nudge for pad / release-hole clearance, then apply collar-orientation shift.


In [ ]:
from IPython.display import HTML, display

from src.prep_resonator_ppd import (
    MIN_RELEASE_HOLE_CLEARANCE_UM,
    assemblies_summary_df,
    prep_resonator_ppd,
)

ppd_assemblies = prep_resonator_ppd(
    res_df,
    res_list,
    PPD,
    identification=identification,
    layermap=layermap,
)
display(assemblies_summary_df(ppd_assemblies))


simply preview of the resonator + ppd GSG placement within the notebook directly 

In [ ]:
# # Display/Preview of frame within this notebook

# _preview_cols = 4
# _preview_items = "\n".join(
#     f'<div class="ppd-preview-item">'
#     f'<div class="ppd-preview-label">[{asm.index}] {asm.inst_name} ({asm.res_type})</div>'
#     f'{preview_assembly_svg(asm)}'
#     f"</div>"
#     for asm in ppd_assemblies
# )
# display(
#     HTML(
#         f"""
# <style>
# .ppd-preview-grid {{
#   display: grid;
#   grid-template-columns: repeat({_preview_cols}, minmax(0, 1fr));
#   gap: 8px;
#   margin-top: 8px;
# }}
# .ppd-preview-item {{
#   border: 1px solid #ccc;
#   padding: 4px;
#   text-align: center;
#   overflow: hidden;
# }}
# .ppd-preview-label {{
#   font-size: 11px;
#   margin-bottom: 4px;
# }}
# .ppd-preview-item svg {{
#   width: 100%;
#   height: auto;
#   display: block;
# }}
# </style>
# <div class="ppd-preview-grid">
# {_preview_items}
# </div>
# """
#     )
# )

## 4. Setting Up

place the ppd+resonator combo now into the original die frame.

for the width (x axis) leave 4% margin in both sides

for the height (y axis) leave 7% margin in both sides

### `prep_rteg_frame.py` — die frame placement

For each `ppd_assembly`, place the step-2 `top_cell` inside `FRAME`:

1. Die frame at `(0, 0)`; margins measured from the **inner** MBE die frame (not the outer 460×580 bbox)
2. Assembly left edge at 3.5% inside the inner frame; Y centered with 7% margins
3. MBE filler rectangle on the right: inner-frame center → margined right edge, same height as the placed GSG/resonator assembly

Returns `rteg_assemblies` and `rteg_files_df`. Step 5 (later) adds interconnect routing and DRC.

In [ ]:
from src.prep_rteg_frame import (
    assemblies_summary_df,
    prep_rteg_in_frame,
    preview_assembly_svg,
)

rteg_assemblies = prep_rteg_in_frame(ppd_assemblies, FRAME)
rteg_files_df = assemblies_summary_df(rteg_assemblies)

print(f"Built {len(rteg_assemblies)} RTEG frame assemblies")
display(rteg_files_df)

In [ ]:
# from IPython.display import HTML, display

# _preview_cols = 4
# _preview_items = "\n".join(
#     f'<div class="rteg-preview-item">'
#     f'<div class="rteg-preview-label">[{asm.index}] {asm.inst_name} ({asm.ppd_assembly.res_type})</div>'
#     f'{preview_assembly_svg(asm)}'
#     f"</div>"
#     for asm in rteg_assemblies
# )
# display(
#     HTML(
#         f"""
# <style>
# .rteg-preview-grid {{
#   display: grid;
#   grid-template-columns: repeat({_preview_cols}, minmax(0, 1fr));
#   gap: 8px;
#   margin-top: 8px;
# }}
# .rteg-preview-item {{
#   border: 1px solid #ccc;
#   padding: 4px;
#   text-align: center;
#   overflow: hidden;
# }}
# .rteg-preview-label {{
#   font-size: 11px;
#   margin-bottom: 4px;
# }}
# .rteg-preview-item svg {{
#   width: 100%;
#   height: auto;
#   display: block;
# }}
# </style>
# <div class="rteg-preview-grid">
# {_preview_items}
# </div>
# """
#     )
# )

In [ ]:
from src.export_gds import export_gds, export_summary_df

rteg_export_df = export_summary_df(
    export_gds(
        rteg_assemblies,
        ORIGNAL_RTEG,
        layermap=layermap,
        parent=parent,
        stage="framed",
    )
)

print(f"Exported {len(rteg_export_df)} GDS files to {ORIGNAL_RTEG}")
print("Layer names: open each .gds with its matching .lyp in KLayout")
display(rteg_export_df)


========================================================

## 5. Routing - solving interconnect algorithm

**Goal:** turn a framed resonator into a DRC-clean RTEG — one fused MBE ground body with two pockets carved out (resonator + signal)

steps to solve this 

1. **Collect (5.1)** — pull ground plates, preserved MBE/MTE, release holes, frame boundary by layermap (`rteg_collect.py`).
2. **Classify (5.2)** — collar orientation, axis, signal vs ground (`rteg_classify.py`).
3. **MTE extensions (5.3)** — draw one ~13 µm extension from the preserved collar that overlaps resonator-body MTE (`build_mte_extensions`).
4. **Route MTE (5.4)** — use 5.2 classification to connect extensions (center pad vs ground); not implemented yet.
5. **Union ground (5.5)** — OR ground node blocks + filler + preserved MBE.
6. **Carve pockets (5.6)** — subtract signal net, resonator keep-out, release holes.
7. **Reconnect (5.7)** — union preserved ground metal back; drop slivers.


### 5.1 — Collect geometry roles

**Files:** `src/rteg_collect.py`

**Entry points:** `collect_geometry_roles`

Splits the framed layout into typed polygon sets (ground plates, preserved metal, release holes, frame boundary). Step 5.2 assigns signal vs ground from collar orientation.


In [ ]:
from src.rteg_collect import (
    RtegCollectConfig,
    collect_geometry_roles,
    geometry_roles_summary_table,
)

COLLECT_CONFIG = RtegCollectConfig()

all_roles: dict[int, object] = {}
collect_rows: list[dict[str, object]] = []
collect_detail_rows: list[dict[str, object]] = []

for idx in range(len(identification.resonators)):
    res = identification.resonators[idx]
    roles = collect_geometry_roles(
        rteg_assemblies[idx],
        res,
        identification,
        layermap,
        config=COLLECT_CONFIG,
    )
    all_roles[idx] = roles
    counts = roles.group_counts()
    collect_rows.append(
        {
            "index": idx,
            "inst_name": roles.inst_name,
            "res_type": res.res_type,
            **{k: counts[k] for k in sorted(counts)},
            "total_polygons": sum(counts.values()),
        }
    )
    collect_detail_rows.extend(geometry_roles_summary_table(roles))

collect_overview_df = pd.DataFrame(collect_rows).sort_values("index")
collect_detail_df = pd.DataFrame(collect_detail_rows)

print(f"Collected geometry for {len(all_roles)} resonators\n")
display(collect_overview_df)
print("\nPolygon detail (all resonators):")
display(collect_detail_df)

### 5.2 — Classify GSG nodes by collar orientation

**Files:** `src/rteg_orientation.py`, `src/rteg_classify.py`, `src/rteg_collect.py`

MTE routing has **two modes only**:
- **`center_pad`** — preserved MTE faces **center** → later routing may connect to center signal pad
- **`collar_extend`** — not facing center → later routing may extend further toward pads

All MTE geometry must launch from **preserved filter MTE** — never from arbitrary resonator metal.

#### Overview table columns

| Column | Meaning |
|--------|---------|
| `mte_route_target` | `center_pad` or `collar_extend` (see above). |
| `mte_faces_center` | `True` when preserved MTE’s nearest GSG band is **center** **and** (east-west) MTE is on the pad side of the body—not directly opposite across the pad–body midpoint in X. |
| `signal_terminal` | `MTE` if preserved filter MTE exists; `MBE` if not. |
| `signal_drawable` | `True` when preserved MTE exists (used by later routing steps). |
| `facing_pad` | Closest GSG band to the collar (`top` / `center` / `bottom`) — geometry only. |
| `collar_axis` | Collar long axis: `east_west` or `north_south`. |
| `top` / `center` / `bottom` | Net assignment per band (`signal` or `ground`). |


In [ ]:
from src.rteg_classify import ClassifyNodesConfig, classify_nodes, classification_summary_table
from src.rteg_collect import collect_orientation_inputs

CLASSIFY_CONFIG = ClassifyNodesConfig()

all_classify: dict[int, object] = {}
classify_overview_rows: list[dict[str, object]] = []
classify_detail_rows: list[dict[str, object]] = []

for idx, roles in all_roles.items():
    res = identification.resonators[idx]
    orientation = collect_orientation_inputs(
        rteg_assemblies[idx],
        res,
        identification,
        layermap,
        ground_plates=roles.ground_plates,
        config=COLLECT_CONFIG,
    )
    classification = classify_nodes(
        roles.ground_plates,
        roles.preserved,
        orientation=orientation,
        res_type=res.res_type,
        config=CLASSIFY_CONFIG,
    )
    all_classify[idx] = classification
    collar = classification.collar_orientation
    by_band = classification.by_band()
    classify_overview_rows.append(
        {
            "index": idx,
            "inst_name": roles.inst_name,
            "res_type": res.res_type,
            "method": classification.method,
            "mte_route_target": classification.mte_route_target,
            "mte_faces_center": collar.mte_faces_center,
            "signal_terminal": classification.signal_terminal,
            "signal_drawable": classification.signal_drawable,
            "collar_axis": collar.axis,
            "facing_pad": collar.facing_pad,
            "top": by_band["top"].net,
            "center": by_band["center"].net,
            "bottom": by_band["bottom"].net,
            "note": classification.note,
        }
    )
    classify_detail_rows.extend(
        classification_summary_table(
            classification,
            index=idx,
            inst_name=roles.inst_name,
            res_type=res.res_type,
        )
    )

classify_overview_df = pd.DataFrame(classify_overview_rows).sort_values("index")
classify_df = pd.DataFrame(classify_detail_rows)

print(f"Classified {len(all_classify)} resonators\n")
display(classify_overview_df)

for idx, classification in all_classify.items():
    assert classification.method == "orientation"
    assert classification.mte_route_target in ("center_pad", "collar_extend")
    assert classification.signal_drawable == bool(all_roles[idx].preserved.mte)
    if classification.mte_route_target == "center_pad":
        assert classification.by_band()["center"].net == "signal"

print(f"\nOrientation classification checks passed for all {len(all_classify)} resonators")


### 5.3 — MTE extensions (one step)

**Files:** `src/rteg_mte_extensions.py`

**Single call:** `build_mte_extensions(all_roles, layermap)`

Step 5.1 often finds two preserved MTE pieces (resonator outline + edge collar). Step 5.3 selects the collar that overlaps resonator-body MTE and draws one new polygon (~13 µm outward, straight open end). Original preserved MTE is not modified.

| Column | Meaning |
|--------|---------|
| `n_preserved_mte` | Preserved MTE count from step 5.1 (often 2) |
| `n_extensions` | Drawn extension polygons (0 or 1 per resonator) |
| `is_connected` | Each extension overlaps its collar |

Writes GDS to `draft_output/MTE_generated_deterministic/` (frame + drawn extensions).


In [ ]:
from export_gds import export_summary_df
from rteg_mte_extensions import (
    build_mte_extensions,
    export_mte_extensions_gds,
    mte_extensions_overview_rows,
    mte_intercept_breakdown_rows,
)

all_mte = build_mte_extensions(all_roles, layermap)

inst_names = {idx: roles.inst_name for idx, roles in all_roles.items()}

mte_overview_df = pd.DataFrame(
    mte_extensions_overview_rows(all_mte, inst_names=inst_names)
).sort_values("index")

display(mte_overview_df)
print(f"Drew MTE extensions for {len(all_mte)} resonators")

mte_intercept_df = pd.DataFrame(
    mte_intercept_breakdown_rows(all_mte, inst_names=inst_names)
).sort_values("index")

print("Intercept breakdown (two end-cap edges on preserved collar):")
display(mte_intercept_df)

MTE_OUT = DRAFT / "MTE_generated_deterministic"
mte_export_df = export_summary_df(
    export_mte_extensions_gds(
        rteg_assemblies,
        all_mte,
        MTE_OUT,
        layermap=layermap,
        parent=parent,
    )
)

print(f"Exported {len(mte_export_df)} MTE GDS files to {MTE_OUT}")


### 5.4 — Route MTE (future)

**Files:** TBD

Uses **5.2** classification (`mte_route_target`, collar axis, `facing_pad`) to decide how each **5.3** extension connects — e.g. center pad vs ground-only extend.

Not implemented yet; 5.3 draws the baseline ~13 µm extension from the collar overlapping resonator-body MTE.
